# Connected hypergraphs: LOMO × Merkle break points × MMR (replayable)

Nodes are content addresses; hyperedges (custody, recipe, amendment, truth, toy, evaluation, consensus, execution, ledger) join many nodes and are hashed. The hypergraph root covers every edge; snapshots are appended to `daisy/hypergraph_mmr.jsonl`. LOMO removes a node (here an actor) and recomputes what depends on it.

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == 'notebooks': os.chdir('..')
sys.path.insert(0, os.getcwd())
from baebugs import hypergraph as HG
g, log = HG.snapshot('.')
print('root', g['root'][:16], g['counts'], 'nodes', len(g['nodes']), '| snapshots', len(log.records), log.verify())

root 66b1f4ef912df5e6 {'amendment': 2, 'consensus': 4, 'custody': 20, 'datasets_root': 1, 'evaluation': 52, 'execution': 1, 'hybrid': 3, 'ledger': 10, 'recipe': 18, 'toy': 91, 'truth': 1} nodes 715 | snapshots 3 (True, None)


## Walk any node back to the frozen truth

In [2]:
for start in ('dataset:multires_mp_1', 'protocol:DAISY-BB-001b'):
    print(start)
    for step in HG.walk_to_truth(g, start): print('   ', step[0], step[1], step[2][:60])

dataset:multires_mp_1
    datasets_root 81abcb94b5f0 datasets_root:5a0d1661538f2a9c1338a274f90477cade19ed034bb09b
    truth 93391532d984 truth:5d190ae6be4b3b6b4aaa5bf32ce030ad2a09d06ae4589e63e2aeb9
protocol:DAISY-BB-001b
    recipe 106817ecfcf5 dataset:synth_1_30_r2
    datasets_root 81abcb94b5f0 datasets_root:5a0d1661538f2a9c1338a274f90477cade19ed034bb09b
    truth 93391532d984 truth:5d190ae6be4b3b6b4aaa5bf32ce030ad2a09d06ae4589e63e2aeb9


## Leave-one-actor-out: which verifier's attestation changes the consensus

In [3]:
for v in ('exact', 'tol_0.5mm', 'plan_only'):
    r = HG.lomo_actors('.', v)
    print(v, r['baseline'])
    for a, x in list(r['leave_one_actor_out'].items())[:5]: print('   ', a, x['flips'], x['examples'][:2])

exact {'FAILED': 5, 'DISPUTED': 86}
    lfm2.5-1.2b-instruct_q4_k_m 0 []
    qwen3_1.7b 0 []
    classical@10px/mm 0 []
    classical@20px/mm 0 []
    classical@5px/mm 0 []
tol_0.5mm {'FAILED': 4, 'SUPPORTED': 15, 'DISPUTED': 72}
    lfm2.5-1.2b-instruct_q4_k_m 6 ['Formicidae_mean_6.9mm', 'Gerridae_mean-1SD_16.2mm']
    qwen3_1.7b 6 ['Formicidae_mean_6.9mm', 'Gerridae_mean-1SD_16.2mm']
    yolo@10px/mm 6 ['Formicidae_mean_6.9mm', 'Gerridae_mean-1SD_16.2mm']
    yolo@5px/mm 6 ['Formicidae_mean_6.9mm', 'Gerridae_mean-1SD_16.2mm']
    yoloe@5px/mm 4 ['Formicidae_mean_6.9mm', 'Halictidae_mean-2SD_2.9mm']
plan_only {'FAILED': 4, 'SUPPORTED': 18, 'DISPUTED': 69}
    yolo@10px/mm 8 ['Andrenidae_mean+2SD_13.6mm', 'Apidae_mean_13.5mm']
    yoloe@5px/mm 8 ['Andrenidae_mean+2SD_13.6mm', 'Apidae_mean_13.5mm']
    yolo@5px/mm 7 ['Andrenidae_mean+2SD_13.6mm', 'Apidae_mean_13.5mm']
    yoloe@20px/mm 7 ['Andrenidae_mean+2SD_13.6mm', 'Apidae_mean_13.5mm']
    lfm2.5-1.2b-instruct_q4_k_m 6 ['Cicadidae_m

## Hybrids: break points from different graphs joined into new ids (proofs re-verified)

In [4]:
from baebugs import hybrid as HY
from baebugs.mmr import MMRLog
for r in MMRLog('daisy/hybrids_mmr.jsonl').records:
    h = r['record']; print(h['id'][:16], h['rule'], '| parents', [p['graph'] for p in h['parents']], '| verify', HY.verify_hybrid(h))

5d4d25dc7ae45ddb incident: failed execution x protocol amendment x recipe | parents ['hypergraph', 'hypergraph', 'impact_mmr'] | verify (True, None)
6c6366c1eb3410e8 toy x filter passage | parents ['passage:body_folded', 'truth:5d190ae6be4b3b6b'] | verify (True, None)
b6f4f8f84d28db5c consensus x leave-one-actor-out | parents ['impact_mmr', 'impact_mmr', 'impact_mmr', 'impact_mmr', 'impact_mmr'] | verify (True, None)


## LOMO x 3: leave one module / one Merkle / one custody break out

In [5]:
r = HG.lomo_graph('.')
for m, x in r['module'].items(): print('MODULE', m, x['invalidated']['by_kind'], x['consensus_after'].get('tol_0.5mm'))
for k, v in sorted(r['merkle'].items(), key=lambda kv: -kv[1]['edges'])[:6]: print('MERKLE', k, v['edges'])
for k, v in sorted(r['custody_break'].items(), key=lambda kv: kv[1]['edges'])[:6]: print('BREAK', k, v['edges'], v['by_kind'])

MODULE insect_lomo {'evaluation': 4, 'consensus': 4, 'hybrid': 1} {'UNVERIFIED_DISPUTED': 76, 'UNVERIFIED_SUPPORTED': 15}
MODULE model_compare.text {'evaluation': 8, 'consensus': 4, 'hybrid': 1} {'DISPUTED': 66, 'SUPPORTED': 21, 'FAILED': 4}
MODULE model_compare.vision {'evaluation': 36, 'consensus': 4, 'hybrid': 1} {'DISPUTED': 83, 'SUPPORTED': 4, 'FAILED': 4}
MODULE toy_triage {'evaluation': 4} {'DISPUTED': 72, 'SUPPORTED': 15, 'FAILED': 4}
MERKLE dataset:gbif_az 172
MERKLE dataset:gbif_ca_2 172
MERKLE dataset:gbif_nv 172
MERKLE dataset:gbif_or 172
MERKLE dataset:gbif_specimen_images 172
MERKLE dataset:synth_1_30_r2 170
BREAK hybrid:6cdcc8cc24 (incident: failed execution x protocol am) 1 {'hybrid': 1}
BREAK hybrid:f405544c3f (toy x filter passage) 1 {'hybrid': 1}
BREAK hybrid:e69eb8fbb4 (consensus x leave-one-actor-out) 1 {'hybrid': 1}
BREAK execution:b18d7c299e 2 {'execution': 1, 'hybrid': 1}
BREAK amendment:3a03c56b4c 7 {'amendment': 1, 'recipe': 6}
BREAK amendment:72b8751201 15 {'